# 8장. 숫자 표와 곱하고 더하기: 텐서와 행렬 곱셈

LLM 안의 계산을 작은 숫자로 직접 해 봅니다. 예문은 "토끼가 숲에 가요"입니다.

- GPU가 없어도 됩니다.
- 셀은 위에서부터 차례로 실행합니다(Shift + Enter).

## 실습 8-1 글자 번호를 텐서로

"토끼가 숲에 가요"의 글자 번호를 텐서에 담습니다.

In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm-v2
import torch
from modu_llm import Vocab

vocab = Vocab.build()
ids = torch.tensor(vocab.encode("토끼가 숲에 가요"))
print(ids)
print(ids.dtype)

## 실습 8-2 모양 읽기

숫자 하나, 숫자 카드 한 장, 숫자 카드 세 장을 쌓은 표를 만들고 모양을 봅니다.

In [ ]:
one = torch.tensor(3.5)                           # 숫자 하나
card = torch.tensor([1.0, 0.0, 2.0, 0.0])         # 숫자 카드 한 장
cards = torch.tensor([[1.0, 0.0, 2.0, 0.0],       # "토"의 숫자 카드
                      [0.0, 1.0, 0.0, 1.0],       # "끼"의 숫자 카드
                      [2.0, 0.0, 0.0, 1.0]])      # "가"의 숫자 카드
print(one.shape)
print(card.shape)
print(cards.shape)
print(ids.shape)

## 실습 8-3 줄과 칸 꺼내기, 모양 바꾸기

**먼저 예상해 보세요.** `cards[:, 0]`에는 어떤 숫자 세 개가 나올까요?

In [ ]:
print(cards[0])          # 0번 줄: "토"의 숫자 카드
print(cards[2, 3])       # 2번 줄의 3번 칸
print(cards[:, 0])       # 모든 줄의 0번 칸
print(ids.view(3, 3))    # 번호 9개를 3줄 × 3칸으로

## 실습 8-4 곱해서 더하기: 내적

"가"의 질문 쪽지를 세 글자의 명찰에 하나씩 맞대어 점수를 매깁니다.

In [ ]:
q3 = torch.tensor([2.0, 0.0, 0.0, 2.0])    # "가"의 질문 쪽지
k1 = torch.tensor([0.0, 0.0, 1.0, 2.0])    # "토"의 명찰
k2 = torch.tensor([1.0, 0.0, 0.0, 0.0])    # "끼"의 명찰
k3 = torch.tensor([0.0, 1.0, 1.0, 0.0])    # "가"의 명찰

print("같은 자리끼리 곱하기:", q3 * k1)
print("모두 더하기:", (q3 * k1).sum())
print("@로 한 번에:", q3 @ k1, q3 @ k2, q3 @ k3)

## 실습 8-5 표 뒤집기: 전치

명찰 세 장을 쌓아 표를 만들고 뒤집어 봅니다.

In [ ]:
K = torch.stack([k1, k2, k3])    # 명찰 세 장을 쌓은 표
print(K)
print(K.T)
print(K.shape, "→", K.T.shape)

## 실습 8-6 표와 표 곱하기: 행렬 곱셈

**먼저 예상해 보세요.** 점수 표 S의 셋째 줄("가"의 질문 쪽지 줄)에는 어떤 숫자 세 개가 들어 있을까요?

In [ ]:
Q = torch.tensor([[0.0, 2.0, 0.0, 0.0],      # "토"의 질문 쪽지
                  [0.0, 0.0, -2.0, 2.0],     # "끼"의 질문 쪽지
                  [2.0, 0.0, 0.0, 2.0]])     # "가"의 질문 쪽지
S = Q @ K.T
print(S)
print(S.shape)

## 실습 8-7 숫자 카드에서 점수 표까지

9글자의 숫자 카드로 질문 쪽지와 명찰을 만들고 점수 표까지 계산합니다. 숫자는 무작위입니다.

In [ ]:
torch.manual_seed(0)
X = torch.randn(9, 8)      # 9글자의 숫자 카드(글자마다 숫자 8개)
Wq = torch.randn(8, 4)     # 질문 쪽지를 만드는 표
Wk = torch.randn(8, 4)     # 명찰을 만드는 표

Q9 = X @ Wq                # 9글자의 질문 쪽지
K9 = X @ Wk                # 9글자의 명찰
S9 = Q9 @ K9.T             # 점수 표
print(Q9.shape, K9.shape, S9.shape)
print("'가'의 점수 줄:", S9[2])
print("한 칸은 내적 하나:", torch.allclose(S9[2, 0], Q9[2] @ K9[0]))